# Genie Validation - Marathos

Testing the Marathos Genie space.

**Approach:** for each question the correct query will be written *manually* and run here. Genie is asked the same question, and its answers will be compared.

Genie space is connected to the whole `marathos.gold` schema (fact, dimensions, mart and views) so that it can answer both broad and Sweden-specific questions. The questions here will/are very much reminiscent of earlier tasks and work.

## Question 1: Which Swedish event has the most finishers?

In [0]:
# My answer:

spark.sql("""
SELECT event_name, COUNT(*) AS n_finishes
FROM marathos.gold.mart_sweden
GROUP BY event_name
ORDER BY n_finishes DESC
LIMIT 5
""").display()

In [0]:
%sql

-- Genie

WITH event_finishers AS (
  SELECT
    `event_name`,
    COUNT(`athlete_id`) AS n_finishers
  FROM
    `marathos`.`gold`.`mart_sweden`
  WHERE
    `event_name` ILIKE '%(SWE)'
    AND `athlete_id` IS NOT NULL
  GROUP BY
    `event_name`
)
SELECT
  `event_name`,
  n_finishers
FROM
  (
    SELECT
      *,
      RANK() OVER (ORDER BY n_finishers DESC) AS rank
    FROM
      event_finishers
  )
WHERE
  rank = 1;

**Question 1:** Same top event (Ultravasan90). Genie returned only #1 via RANK()=1; while I showed top 5. It added a redundant `(SWE)` filter and used COUNT(athlete_id) but same result.

## Q2: What is the gender distribution across all finishes?

In [0]:
# My answer:

spark.sql("""
SELECT a.athlete_gender, COUNT(*) AS n_finishes
FROM marathos.gold.fct_results f
JOIN marathos.gold.dim_athlete a ON f.athlete_id = a.athlete_id
GROUP BY a.athlete_gender
ORDER BY n_finishes DESC
""").display()

In [0]:
%sql

-- Genie

SELECT
  a.athlete_gender,
  COUNT(*) AS n_finishes
FROM
  `marathos`.`gold`.`fct_results` r
    JOIN `marathos`.`gold`.`dim_athlete` a
      ON r.athlete_id = a.athlete_id
WHERE
  a.athlete_gender IS NOT NULL
GROUP BY
  a.athlete_gender
ORDER BY
  n_finishes DESC

**Q2:** Identical query. Genie added `WHERE athlete_gender IS NOT NULL`, which changes nothing (silver has no nulls, 'unknown' still counted). Same result.



## Q3: Which country has the most finishes?

In [0]:
# My answer

spark.sql("""
SELECT c.country_name, COUNT(*) AS n_finishes
FROM marathos.gold.fct_results f
JOIN marathos.gold.dim_athlete a ON f.athlete_id = a.athlete_id
JOIN marathos.gold.dim_country c ON a.athlete_country = c.country_code
GROUP BY c.country_name
ORDER BY n_finishes DESC
LIMIT 10
""").display()

In [0]:
%sql

-- Genie

WITH distance_finishes AS (
  SELECT
    athlete_country,
    n_finishes
  FROM
    `marathos`.`gold`.`view_distance_country_stats`
),
length_finishes AS (
  SELECT
    athlete_country,
    n_finishes
  FROM
    `marathos`.`gold`.`view_length_country_stats`
),
total_finishes AS (
  SELECT
    athlete_country,
    n_finishes
  FROM
    distance_finishes
  UNION ALL
  SELECT
    athlete_country,
    n_finishes
  FROM
    length_finishes
)
SELECT
  c.country_name,
  SUM(tf.n_finishes) AS total_finishes
FROM
  total_finishes tf
    JOIN `marathos`.`gold`.`dim_country` c
      ON tf.athlete_country = c.country_code
WHERE
  c.country_name IS NOT NULL
GROUP BY
  c.country_name
ORDER BY
  total_finishes DESC
LIMIT 5

**Q3:** Different path, same result. Genie summed the two country-stats views (UNION ALL) instead of the fact. Correct because distance + length covers all finishes.


## Q4: How has the number of finishes changed over the years?

In [0]:
# My answer

spark.sql("""
SELECT d.year, COUNT(*) AS n_finishes
FROM marathos.gold.fct_results f
JOIN marathos.gold.dim_date d ON f.date_id = d.date_id
GROUP BY d.year
ORDER BY d.year
""").display()

In [0]:
%sql

-- Genie

SELECT
  d.year,
  COUNT(r.result_id) AS n_finishes
FROM
  `marathos`.`gold`.`fct_results` r
    JOIN `marathos`.`gold`.`dim_date` d
      ON r.date_id = d.date_id
WHERE
  d.year IS NOT NULL
GROUP BY
  d.year
ORDER BY
  d.year ASC

**Q4:** Near-identical. Genie used COUNT(result_id) and `WHERE year IS NOT NULL`. No null years exist. Same result.


# Short Summary:

All four answers from genie matched mine. However, Genie did consistently add null guards that were unnecessary, which confirms that the silver cleanup already removed null values. The only substantial deviation here was 'Question 3', where Genie reused gold views instead of the fact table and still landed correctly, since distance + length covers the entire data. Writing the queries manually first made these differences quite easy to spot and verify.